# Timeframe Analysis: Cointegration Across Different Intervals

**Goal**: Test if cointegration is timeframe-dependent by analyzing pairs at different intervals.

## Why This Matters

From our previous analysis, we noticed pairs appearing/disappearing from cointegration tests:
- **SUSHI/UNI**: Was cointegrated (p=0.032), now not (p=0.091)
- **CRV/SNX**: Was cointegrated (p=0.041), now not (p=0.405)

This suggests:
1. **Time-dependent relationships**: Cointegration changes over time
2. **Timeframe sensitivity**: Different intervals may show different relationships
3. **Need for stability checks**: Rolling window analysis is essential

## Timeframes to Test

- **15 minutes** (15m) - High frequency, ~6,000 candles in 60 days
- **1 hour** (1h) - Medium frequency, ~2,000 candles in 90 days (current)
- **4 hours** (4h) - Lower frequency, ~500 candles in 90 days
- **Daily** (1d) - Long-term, ~90 candles in 90 days

**Note**: We'll start with 15m data for our top 10 currently cointegrated pairs.

In [ ]:
import sys
sys.path.append('..')

import polars as pl
from datetime import datetime, timedelta
from statistical_arbitrage.data.bitvavo_client import BitvavoDataCollector
from statistical_arbitrage.analysis.cointegration import PairAnalysis
import numpy as np

# Enable interactive tables
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

In [ ]:
# Initialize data collector
collector = BitvavoDataCollector()

## Define Top 10 Cointegrated Pairs

These are the pairs that showed cointegration at 1h timeframe:

In [ ]:
# Top 10 cointegrated pairs from hourly analysis
top_pairs = [
    ("PEPE/EUR", "BONK/EUR", "Memecoins"),
    ("WIF/EUR", "BONK/EUR", "Memecoins"),
    ("LINK/EUR", "GRT/EUR", "DeFi"),
    ("ETH/EUR", "ADA/EUR", "Major Coins"),
    ("SHIB/EUR", "BONK/EUR", "Memecoins"),
    ("ETH/EUR", "ETC/EUR", "Major Coins"),
    ("AXS/EUR", "SAND/EUR", "Gaming"),
    ("LINK/EUR", "UNI/EUR", "DeFi"),
    ("FTT/EUR", "CRO/EUR", "Exchange"),
    ("CRV/EUR", "AAVE/EUR", "DeFi"),
]

print(f"Testing {len(top_pairs)} pairs across multiple timeframes")
for asset1, asset2, category in top_pairs:
    print(f"  - {asset1} vs {asset2} ({category})")

## Test Function

In [ ]:
def test_pair_timeframe(asset1: str, asset2: str, interval: str, days_back: int = 60) -> dict:
    """Test cointegration for a pair at a specific timeframe."""
    try:
        # Fetch data
        data1 = collector.get_candles_range(market=asset1, interval=interval, days_back=days_back)
        data2 = collector.get_candles_range(market=asset2, interval=interval, days_back=days_back)
        
        # Align data
        merged = data1.join(data2, on="datetime", how="inner", suffix="_2")
        
        if len(merged) < 30:  # Need minimum data points
            return {
                "error": f"Insufficient data: {len(merged)} candles",
                "data_points": len(merged),
            }
        
        # Extract prices
        prices1 = merged["close"]
        prices2 = merged["close_2"]
        
        # Run analysis
        analysis = PairAnalysis(prices1, prices2)
        coint_results = analysis.test_cointegration()
        
        # Calculate correlation
        correlation = np.corrcoef(prices1.to_numpy(), prices2.to_numpy())[0, 1]
        
        return {
            "data_points": len(merged),
            "correlation": correlation,
            "p_value": coint_results["p_value"],
            "is_cointegrated": coint_results["is_cointegrated"],
            "hedge_ratio": coint_results["hedge_ratio"],
            "half_life": analysis.calculate_half_life() if coint_results["is_cointegrated"] else None,
            "error": None,
        }
    except Exception as e:
        return {
            "error": str(e),
            "data_points": 0,
        }

## Test 15-Minute Timeframe

Let's start with 15-minute data (60 days = ~5,760 candles):

In [ ]:
print("Testing pairs at 15-minute timeframe...\n")

results_15m = []

for asset1, asset2, category in top_pairs:
    print(f"Testing {asset1} vs {asset2}...", end=" ")
    
    result = test_pair_timeframe(asset1, asset2, interval="15m", days_back=60)
    
    if result.get("error"):
        print(f"❌ Error: {result['error']}")
    else:
        status = "✅" if result["is_cointegrated"] else "❌"
        print(f"{status} p={result['p_value']:.4f}, n={result['data_points']}")
    
    result["asset1"] = asset1
    result["asset2"] = asset2
    result["category"] = category
    result["timeframe"] = "15m"
    results_15m.append(result)

print(f"\nCompleted 15m testing")

## Results: 15-Minute Timeframe

In [ ]:
# Create results dataframe
df_15m = pl.DataFrame([r for r in results_15m if not r.get("error")])

if len(df_15m) > 0:
    display_15m = df_15m.select([
        "category",
        "asset1",
        "asset2",
        "is_cointegrated",
        "p_value",
        "correlation",
        "data_points",
        "half_life",
    ]).sort("p_value")
    
    display_15m
else:
    print("No successful results")

In [ ]:
display_15m

In [ ]:
# Summary statistics
if len(df_15m) > 0:
    cointegrated_15m = df_15m.filter(pl.col("is_cointegrated").cast(pl.Boolean)).shape[0]
    total_15m = df_15m.shape[0]
    print(f"\n15-Minute Timeframe: {cointegrated_15m}/{total_15m} pairs cointegrated")
    print(f"Cointegration rate: {cointegrated_15m/total_15m*100:.1f}%")

## Test 4-Hour Timeframe

Now let's test with 4-hour data for comparison:

In [ ]:
print("Testing pairs at 4-hour timeframe...\n")

results_4h = []

for asset1, asset2, category in top_pairs:
    print(f"Testing {asset1} vs {asset2}...", end=" ")
    
    result = test_pair_timeframe(asset1, asset2, interval="4h", days_back=90)
    
    if result.get("error"):
        print(f"❌ Error: {result['error']}")
    else:
        status = "✅" if result["is_cointegrated"] else "❌"
        print(f"{status} p={result['p_value']:.4f}, n={result['data_points']}")
    
    result["asset1"] = asset1
    result["asset2"] = asset2
    result["category"] = category
    result["timeframe"] = "4h"
    results_4h.append(result)

print(f"\nCompleted 4h testing")

In [ ]:
# Create results dataframe
df_4h = pl.DataFrame([r for r in results_4h if not r.get("error")])

if len(df_4h) > 0:
    display_4h = df_4h.select([
        "category",
        "asset1",
        "asset2",
        "is_cointegrated",
        "p_value",
        "correlation",
        "data_points",
        "half_life",
    ]).sort("p_value")
    
    display_4h
else:
    print("No successful results")

In [ ]:
display_4h

In [ ]:
# Summary statistics
if len(df_4h) > 0:
    cointegrated_4h = df_4h.filter(pl.col("is_cointegrated").cast(pl.Boolean)).shape[0]
    total_4h = df_4h.shape[0]
    print(f"\n4-Hour Timeframe: {cointegrated_4h}/{total_4h} pairs cointegrated")
    print(f"Cointegration rate: {cointegrated_4h/total_4h*100:.1f}%")

## Comparison Across Timeframes

Let's compare which pairs are consistently cointegrated:

## Test Daily Timeframe

Finally, let's test with daily data:

In [ ]:
print("Testing pairs at daily timeframe...\n")

results_1d = []

for asset1, asset2, category in top_pairs:
    print(f"Testing {asset1} vs {asset2}...", end=" ")
    
    result = test_pair_timeframe(asset1, asset2, interval="1d", days_back=365)
    
    if result.get("error"):
        print(f"❌ Error: {result['error']}")
    else:
        status = "✅" if result["is_cointegrated"] else "❌"
        print(f"{status} p={result['p_value']:.4f}, n={result['data_points']}")
    
    result["asset1"] = asset1
    result["asset2"] = asset2
    result["category"] = category
    result["timeframe"] = "1d"
    results_1d.append(result)

print(f"\nCompleted 1d testing")

In [ ]:
# Create results dataframe
df_1d = pl.DataFrame([r for r in results_1d if not r.get("error")])

if len(df_1d) > 0:
    display_1d = df_1d.select([
        "category",
        "asset1",
        "asset2",
        "is_cointegrated",
        "p_value",
        "correlation",
        "data_points",
        "half_life",
    ]).sort("p_value")
    
    display_1d
else:
    print("No successful results")

In [ ]:
# Summary statistics
if len(df_1d) > 0:
    cointegrated_1d = df_1d.filter(pl.col("is_cointegrated").cast(pl.Boolean)).shape[0]
    total_1d = df_1d.shape[0]
    print(f"\nDaily Timeframe: {cointegrated_1d}/{total_1d} pairs cointegrated")
    print(f"Cointegration rate: {cointegrated_1d/total_1d*100:.1f}%")

In [ ]:
# Combine results for comparison
comparison_data = []

for asset1, asset2, category in top_pairs:
    row = {
        "category": category,
        "asset1": asset1,
        "asset2": asset2,
    }
    
    # 15m results
    result_15m = next((r for r in results_15m if r["asset1"] == asset1 and r["asset2"] == asset2), None)
    if result_15m and not result_15m.get("error"):
        row["15m_p_value"] = result_15m["p_value"]
        row["15m_cointegrated"] = result_15m["is_cointegrated"]
    else:
        row["15m_p_value"] = None
        row["15m_cointegrated"] = False
    
    # 1h results (from previous run - all were cointegrated)
    row["1h_cointegrated"] = True
    
    # 4h results
    result_4h = next((r for r in results_4h if r["asset1"] == asset1 and r["asset2"] == asset2), None)
    if result_4h and not result_4h.get("error"):
        row["4h_p_value"] = result_4h["p_value"]
        row["4h_cointegrated"] = result_4h["is_cointegrated"]
    else:
        row["4h_p_value"] = None
        row["4h_cointegrated"] = False
    
    # 1d results
    result_1d = next((r for r in results_1d if r["asset1"] == asset1 and r["asset2"] == asset2), None)
    if result_1d and not result_1d.get("error"):
        row["1d_p_value"] = result_1d["p_value"]
        row["1d_cointegrated"] = result_1d["is_cointegrated"]
    else:
        row["1d_p_value"] = None
        row["1d_cointegrated"] = False
    
    comparison_data.append(row)

comparison_df = pl.DataFrame(comparison_data)
comparison_df

In [ ]:
# Find consistently cointegrated pairs (cointegrated across ALL timeframes)
print("\n" + "="*80)
print("CONSISTENTLY COINTEGRATED PAIRS (across all 4 timeframes)")
print("="*80)

consistent = comparison_df.filter(
    pl.col("15m_cointegrated").cast(pl.Boolean) & 
    pl.col("1h_cointegrated") & 
    pl.col("4h_cointegrated").cast(pl.Boolean) &
    pl.col("1d_cointegrated").cast(pl.Boolean)
)

if len(consistent) > 0:
    print(f"\nFound {len(consistent)} consistently cointegrated pairs:")
    for row in consistent.iter_rows(named=True):
        print(f"  ✅ {row['asset1']} vs {row['asset2']} ({row['category']})")
        print(f"     15m: p={row['15m_p_value']:.4f}, 4h: p={row['4h_p_value']:.4f}, 1d: p={row['1d_p_value']:.4f}")
else:
    print("\n⚠️  No pairs are consistently cointegrated across ALL timeframes!")
    print("This suggests cointegration is HIGHLY TIMEFRAME-DEPENDENT.")

## Key Insights

From this analysis, we can determine:

1. **Timeframe Sensitivity**: Are the same pairs cointegrated at different timeframes?
2. **Stability**: Pairs cointegrated across multiple timeframes are more reliable
3. **Trading Strategy**: Should match the cointegration timeframe to the trading timeframe

## Next Steps

1. **Rolling Window Analysis**: Check if cointegration is stable over time (not just across timeframes)
2. **Half-Life Analysis**: Compare mean reversion speeds across timeframes
3. **Backtest Only Stable Pairs**: Focus on pairs that show consistent cointegration